In [1]:
import os
import numpy as np
import pandas as pd
import pickle

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

pd.set_option("display.max_rows", None, "display.max_columns", None)

RANDOM_STATE = 42

# Voting & Stacking Classifier Comparison
Compare 4 ensemble strategies using pre-trained models:
1. **Voting (all 8)** — soft voting, equal weights
2. **Voting (drop KNN)** — 7 models, equal weights  
3. **Weighted Voting (all 8)** — weights by inverse log loss
4. **Weighted Voting (drop KNN)** — 7 models, weighted
5. **Stacking (all 8)** — LogReg meta-learner
6. **Stacking (drop KNN)** — LogReg meta-learner, 7 models

In [5]:
# === Configuration ===
# Set this to test men's or women's
IS_WOMENS = True

gender = 'women' if IS_WOMENS else 'men'
model_dir = os.path.abspath(f'../model/{"womens" if IS_WOMENS else "mens"}/')

# Model name -> pkl filename mapping
MODEL_FILES = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}

def get_datasets(span):
    path = os.path.abspath(f'../../data/dataset/{gender}/{span}span_training_set.csv')
    training_df = pd.read_csv(path)
    path = os.path.abspath(f'../../data/dataset/{gender}/{span}span_testing_set.csv')
    testing_df = pd.read_csv(path)
    train_true, test_true = training_df.pop('Win'), testing_df.pop('Win')
    return training_df, testing_df, train_true, test_true

def load_models(span):
    """Load all pre-trained models for a given span."""
    models = {}
    for name, fname in MODEL_FILES.items():
        path = os.path.join(model_dir, f'{span}span_{fname}')
        models[name] = pickle.load(open(path, 'rb'))
    return models

def evaluate(clf, testing_df, test_true, label=""):
    """Evaluate a classifier and return metrics dict."""
    y_pred = clf.predict(testing_df)
    y_proba = clf.predict_proba(testing_df)[:, 1]
    acc = accuracy_score(test_true, y_pred) * 100
    ll = log_loss(test_true, y_proba)
    bs = brier_score_loss(test_true, y_proba)
    return {'label': label, 'accuracy': round(acc, 2), 'log_loss': round(ll, 4), 'brier_score': round(bs, 4)}

print(f"Configured for: {'Women' if IS_WOMENS else 'Men'}'s basketball")
print(f"Model directory: {model_dir}")
print(f"Models: {list(MODEL_FILES.keys())}")

Configured for: Women's basketball
Model directory: /workspace/machine-learning/model/womens
Models: ['logistic_regression', 'svm', 'knn', 'random_forest', 'gradient_boosting', 'mlp', 'xgboost', 'lightgbm']


In [6]:
# === Evaluate all ensemble strategies across all spans ===

all_results = []

# Pre-build a fitted LabelEncoder for binary classes
le = LabelEncoder()
le.fit([0, 1])

for span in [3, 5, 7]:
    print(f'\n{"="*70}')
    print(f'SPAN {span}')
    print(f'{"="*70}')
    
    training_df, testing_df, train_true, test_true = get_datasets(span)
    models = load_models(span)
    
    # --- Individual model log losses (needed for weights) ---
    model_logloss = {}
    for name, model in models.items():
        y_proba = model.predict_proba(testing_df)[:, 1]
        model_logloss[name] = log_loss(test_true, y_proba)
    
    # --- Build estimator lists ---
    all_estimators = [(name, model) for name, model in models.items()]
    no_knn_estimators = [(name, model) for name, model in models.items() if name != 'knn']
    
    # --- Inverse log loss weights ---
    all_weights = [1.0 / model_logloss[name] for name, _ in all_estimators]
    no_knn_weights = [1.0 / model_logloss[name] for name, _ in no_knn_estimators]
    
    print(f"\nModel log losses (for weighting):")
    for name in sorted(model_logloss, key=model_logloss.get):
        w = 1.0 / model_logloss[name]
        print(f"  {name:<25} LL={model_logloss[name]:.4f}  weight={w:.3f}")
    
    # =========================================================
    # 1. Voting - All 8, equal weights
    # =========================================================
    vc_all = VotingClassifier(estimators=all_estimators, voting='soft')
    vc_all.estimators_ = [model for _, model in all_estimators]
    vc_all.le_ = le
    vc_all.classes_ = np.array([0, 1])
    r = evaluate(vc_all, testing_df, test_true, "Voting (all 8, equal)")
    r['span'] = span
    all_results.append(r)
    print(f"\n1. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")
    
    # =========================================================
    # 2. Voting - Drop KNN, equal weights
    # =========================================================
    vc_no_knn = VotingClassifier(estimators=no_knn_estimators, voting='soft')
    vc_no_knn.estimators_ = [model for _, model in no_knn_estimators]
    vc_no_knn.le_ = le
    vc_no_knn.classes_ = np.array([0, 1])
    r = evaluate(vc_no_knn, testing_df, test_true, "Voting (no KNN, equal)")
    r['span'] = span
    all_results.append(r)
    print(f"2. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")
    
    # =========================================================
    # 3. Voting - All 8, weighted by inverse log loss
    # =========================================================
    vc_all_w = VotingClassifier(estimators=all_estimators, voting='soft', weights=all_weights)
    vc_all_w.estimators_ = [model for _, model in all_estimators]
    vc_all_w.le_ = le
    vc_all_w.classes_ = np.array([0, 1])
    r = evaluate(vc_all_w, testing_df, test_true, "Voting (all 8, weighted)")
    r['span'] = span
    all_results.append(r)
    print(f"3. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")
    
    # =========================================================
    # 4. Voting - Drop KNN, weighted
    # =========================================================
    vc_no_knn_w = VotingClassifier(estimators=no_knn_estimators, voting='soft', weights=no_knn_weights)
    vc_no_knn_w.estimators_ = [model for _, model in no_knn_estimators]
    vc_no_knn_w.le_ = le
    vc_no_knn_w.classes_ = np.array([0, 1])
    r = evaluate(vc_no_knn_w, testing_df, test_true, "Voting (no KNN, weighted)")
    r['span'] = span
    all_results.append(r)
    print(f"4. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")
    
    # =========================================================
    # 5. Stacking - All 8, LogReg meta-learner
    # =========================================================
    stack_all = StackingClassifier(
        estimators=all_estimators,
        final_estimator=LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        cv=5,
        stack_method='predict_proba',
        passthrough=False
    )
    stack_all.fit(training_df, train_true)
    r = evaluate(stack_all, testing_df, test_true, "Stacking (all 8)")
    r['span'] = span
    all_results.append(r)
    print(f"5. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")
    
    # =========================================================
    # 6. Stacking - Drop KNN, LogReg meta-learner
    # =========================================================
    stack_no_knn = StackingClassifier(
        estimators=no_knn_estimators,
        final_estimator=LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        cv=5,
        stack_method='predict_proba',
        passthrough=False
    )
    stack_no_knn.fit(training_df, train_true)
    r = evaluate(stack_no_knn, testing_df, test_true, "Stacking (no KNN)")
    r['span'] = span
    all_results.append(r)
    print(f"6. {r['label']:<35} Acc={r['accuracy']:.2f}%  LL={r['log_loss']:.4f}  Brier={r['brier_score']:.4f}")


SPAN 3

Model log losses (for weighting):
  logistic_regression       LL=0.5207  weight=1.920
  svm                       LL=0.5235  weight=1.910
  xgboost                   LL=0.5264  weight=1.900
  lightgbm                  LL=0.5275  weight=1.896
  gradient_boosting         LL=0.5279  weight=1.894
  mlp                       LL=0.5302  weight=1.886
  random_forest             LL=0.5381  weight=1.859
  knn                       LL=0.6216  weight=1.609

1. Voting (all 8, equal)               Acc=73.20%  LL=0.5217  Brier=0.1749
2. Voting (no KNN, equal)              Acc=72.93%  LL=0.5210  Brier=0.1748
3. Voting (all 8, weighted)            Acc=73.08%  LL=0.5214  Brier=0.1748
4. Voting (no KNN, weighted)           Acc=72.93%  LL=0.5210  Brier=0.1748
5. Stacking (all 8)                    Acc=73.17%  LL=0.5242  Brier=0.1758
6. Stacking (no KNN)                   Acc=73.01%  LL=0.5249  Brier=0.1760

SPAN 5

Model log losses (for weighting):
  logistic_regression       LL=0.5180  weight=1

In [7]:
# === Summary table ===
results_df = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("FULL RESULTS TABLE")
print("=" * 80)
print(results_df[['span', 'label', 'accuracy', 'log_loss', 'brier_score']].to_string(index=False))

# Average across spans
print("\n" + "=" * 80)
print("AVERAGE ACROSS SPANS (ranked by log loss)")
print("=" * 80)
avg_results = results_df.groupby('label').agg({
    'accuracy': 'mean',
    'log_loss': 'mean', 
    'brier_score': 'mean'
}).sort_values('log_loss')
print(avg_results.to_string())

# Also compare to best individual model
print("\n" + "=" * 80)
print("FOR REFERENCE — Best individual models (avg across spans):")
print("=" * 80)
for span in [3, 5, 7]:
    training_df, testing_df, train_true, test_true = get_datasets(span)
    models = load_models(span)
    for name, model in models.items():
        y_proba = model.predict_proba(testing_df)[:, 1]
        y_pred = model.predict(testing_df)
        all_results.append({
            'label': f'[Individual] {name}',
            'span': span,
            'accuracy': round(accuracy_score(test_true, y_pred) * 100, 2),
            'log_loss': round(log_loss(test_true, y_proba), 4),
            'brier_score': round(brier_score_loss(test_true, y_proba), 4)
        })

ind_df = pd.DataFrame([r for r in all_results if r['label'].startswith('[Individual]')])
ind_avg = ind_df.groupby('label').agg({
    'accuracy': 'mean',
    'log_loss': 'mean',
    'brier_score': 'mean'
}).sort_values('log_loss')
print(ind_avg.to_string())


FULL RESULTS TABLE
 span                     label  accuracy  log_loss  brier_score
    3     Voting (all 8, equal)     73.20    0.5217       0.1749
    3    Voting (no KNN, equal)     72.93    0.5210       0.1748
    3  Voting (all 8, weighted)     73.08    0.5214       0.1748
    3 Voting (no KNN, weighted)     72.93    0.5210       0.1748
    3          Stacking (all 8)     73.17    0.5242       0.1758
    3         Stacking (no KNN)     73.01    0.5249       0.1760
    5     Voting (all 8, equal)     74.04    0.5187       0.1739
    5    Voting (no KNN, equal)     73.73    0.5185       0.1739
    5  Voting (all 8, weighted)     73.98    0.5186       0.1738
    5 Voting (no KNN, weighted)     73.74    0.5185       0.1739
    5          Stacking (all 8)     73.97    0.5204       0.1741
    5         Stacking (no KNN)     73.83    0.5212       0.1743
    7     Voting (all 8, equal)     74.28    0.5077       0.1698
    7    Voting (no KNN, equal)     74.23    0.5070       0.1698
    7

In [3]:
# === Probability Trimming Analysis ===
# Test whether clipping probabilities to [ε, 1-ε] improves metrics

epsilons = [0, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]

for gender_label, is_womens in [("MEN'S", False), ("WOMEN'S", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')
    
    print(f"\n{'='*80}")
    print(f"  PROBABILITY TRIMMING — {gender_label}")
    print(f"{'='*80}")
    
    for span in [3, 5, 7]:
        # Load data
        tr_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_training_set.csv')
        te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
        training_df = pd.read_csv(tr_path)
        testing_df = pd.read_csv(te_path)
        train_true, test_true = training_df.pop('Win'), testing_df.pop('Win')
        
        # Load models
        model_files = {
            'logistic_regression': 'logistic_regression_model.pkl',
            'svm': 'support_vector_machine_model.pkl',
            'knn': 'knn_model.pkl',
            'random_forest': 'random_forest.pkl',
            'gradient_boosting': 'gradient_boosting.pkl',
            'mlp': 'multilayer_perceptron.pkl',
            'xgboost': 'xgboost.pkl',
            'lightgbm': 'lightgbm.pkl'
        }
        models = {}
        for name, fname in model_files.items():
            path = os.path.join(mdir, f'{span}span_{fname}')
            models[name] = pickle.load(open(path, 'rb'))
        
        print(f"\n  SPAN {span}")
        print(f"  {'Model':<25} | " + " | ".join(f"ε={e:<4}" for e in epsilons))
        print(f"  {'-'*25}-+-" + "-+-".join("-"*7 for _ in epsilons))
        
        for name, model in models.items():
            y_proba = model.predict_proba(testing_df)[:, 1]
            
            # Show log loss across epsilons
            lls = []
            for eps in epsilons:
                clipped = np.clip(y_proba, eps, 1 - eps)
                lls.append(log_loss(test_true, clipped))
            print(f"  {name:<25} | " + " | ".join(f"{ll:.4f}" for ll in lls))
        
        # Also test ensemble (soft vote, no KNN, equal weights)
        no_knn_models = {k: v for k, v in models.items() if k != 'knn'}
        avg_proba = np.mean([m.predict_proba(testing_df)[:, 1] for m in no_knn_models.values()], axis=0)
        lls = []
        for eps in epsilons:
            clipped = np.clip(avg_proba, eps, 1 - eps)
            lls.append(log_loss(test_true, clipped))
        print(f"  {'>> Ensemble (7, equal)':<25} | " + " | ".join(f"{ll:.4f}" for ll in lls))

    # Summary: find optimal epsilon per model (averaged across spans)
    print(f"\n  {'='*60}")
    print(f"  OPTIMAL EPSILON PER MODEL (avg across spans)")
    print(f"  {'='*60}")
    
    all_model_names = list(model_files.keys()) + ['ensemble_7_equal']
    
    for model_name in all_model_names:
        avg_lls = {eps: 0 for eps in epsilons}
        avg_briers = {eps: 0 for eps in epsilons}
        
        for span in [3, 5, 7]:
            tr_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_training_set.csv')
            te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
            training_df = pd.read_csv(tr_path)
            testing_df = pd.read_csv(te_path)
            train_true, test_true = training_df.pop('Win'), testing_df.pop('Win')
            
            models_span = {}
            for nm, fn in model_files.items():
                path = os.path.join(mdir, f'{span}span_{fn}')
                models_span[nm] = pickle.load(open(path, 'rb'))
            
            if model_name == 'ensemble_7_equal':
                no_knn = {k: v for k, v in models_span.items() if k != 'knn'}
                y_proba = np.mean([m.predict_proba(testing_df)[:, 1] for m in no_knn.values()], axis=0)
            else:
                y_proba = models_span[model_name].predict_proba(testing_df)[:, 1]
            
            for eps in epsilons:
                clipped = np.clip(y_proba, eps, 1 - eps)
                avg_lls[eps] += log_loss(test_true, clipped) / 3
                avg_briers[eps] += brier_score_loss(test_true, clipped) / 3
        
        best_ll_eps = min(avg_lls, key=avg_lls.get)
        best_br_eps = min(avg_briers, key=avg_briers.get)
        
        ll_no_clip = avg_lls[0]
        ll_best = avg_lls[best_ll_eps]
        br_no_clip = avg_briers[0]
        br_best = avg_briers[best_br_eps]
        
        print(f"  {model_name:<25}  Best LL ε={best_ll_eps:.2f} ({ll_no_clip:.4f} → {ll_best:.4f}, Δ={ll_best-ll_no_clip:+.4f})  "
              f"Best Brier ε={best_br_eps:.2f} ({br_no_clip:.4f} → {br_best:.4f}, Δ={br_best-br_no_clip:+.4f})")


  PROBABILITY TRIMMING — MEN'S

  SPAN 3
  Model                     | ε=0    | ε=0.01 | ε=0.02 | ε=0.03 | ε=0.05 | ε=0.08 | ε=0.1  | ε=0.15 | ε=0.2 
  --------------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------
  logistic_regression       | 0.5684 | 0.5685 | 0.5685 | 0.5686 | 0.5688 | 0.5693 | 0.5699 | 0.5726 | 0.5775
  svm                       | 0.5743 | 0.5743 | 0.5743 | 0.5744 | 0.5746 | 0.5752 | 0.5757 | 0.5778 | 0.5819
  knn                       | 0.6439 | 0.6282 | 0.6280 | 0.6279 | 0.6279 | 0.6280 | 0.6282 | 0.6289 | 0.6305
  random_forest             | 0.5872 | 0.5872 | 0.5872 | 0.5872 | 0.5872 | 0.5874 | 0.5876 | 0.5888 | 0.5918
  gradient_boosting         | 0.5735 | 0.5735 | 0.5735 | 0.5736 | 0.5737 | 0.5740 | 0.5745 | 0.5772 | 0.5820
  mlp                       | 0.5780 | 0.5780 | 0.5780 | 0.5780 | 0.5780 | 0.5783 | 0.5786 | 0.5811 | 0.5858
  xgboost                   | 0.5708 | 0.5708 | 0.5708 | 0.5708 | 0.5709 | 0.

In [4]:
# === Compact Trimming Summary ===
# Re-compute just the optimal epsilon table for quick reference

model_files = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}
epsilons = [0, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
all_model_names = list(model_files.keys()) + ['ensemble_7_equal']

rows = []
for gender_label, is_womens in [("Men's", False), ("Women's", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')
    
    for model_name in all_model_names:
        avg_lls = {eps: 0.0 for eps in epsilons}
        avg_briers = {eps: 0.0 for eps in epsilons}
        avg_accs = {eps: 0.0 for eps in epsilons}
        
        for span in [3, 5, 7]:
            te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
            testing_df = pd.read_csv(te_path)
            test_true = testing_df.pop('Win')
            
            models_span = {}
            for nm, fn in model_files.items():
                path = os.path.join(mdir, f'{span}span_{fn}')
                models_span[nm] = pickle.load(open(path, 'rb'))
            
            if model_name == 'ensemble_7_equal':
                no_knn = {k: v for k, v in models_span.items() if k != 'knn'}
                y_proba = np.mean([m.predict_proba(testing_df)[:, 1] for m in no_knn.values()], axis=0)
            else:
                y_proba = models_span[model_name].predict_proba(testing_df)[:, 1]
            
            for eps in epsilons:
                clipped = np.clip(y_proba, eps, 1 - eps)
                avg_lls[eps] += log_loss(test_true, clipped) / 3
                avg_briers[eps] += brier_score_loss(test_true, clipped) / 3
                avg_accs[eps] += accuracy_score(test_true, (clipped >= 0.5).astype(int)) * 100 / 3
        
        best_ll_eps = min(avg_lls, key=avg_lls.get)
        best_br_eps = min(avg_briers, key=avg_briers.get)
        
        rows.append({
            'Gender': gender_label,
            'Model': model_name,
            'LL (ε=0)': round(avg_lls[0], 4),
            'Best LL ε': best_ll_eps,
            'Best LL': round(avg_lls[best_ll_eps], 4),
            'LL Δ': round(avg_lls[best_ll_eps] - avg_lls[0], 4),
            'Brier (ε=0)': round(avg_briers[0], 4),
            'Best Brier ε': best_br_eps,
            'Best Brier': round(avg_briers[best_br_eps], 4),
            'Brier Δ': round(avg_briers[best_br_eps] - avg_briers[0], 4),
            'Acc (ε=0)': round(avg_accs[0], 2),
            'Acc (best LL ε)': round(avg_accs[best_ll_eps], 2),
        })

summary_df = pd.DataFrame(rows)

print("=" * 100)
print("PROBABILITY TRIMMING SUMMARY — Optimal ε per model (avg across spans 3/5/7)")
print("=" * 100)

for gender_label in ["Men's", "Women's"]:
    print(f"\n  {gender_label.upper()} BASKETBALL")
    print(f"  {'-'*90}")
    gdf = summary_df[summary_df['Gender'] == gender_label]
    print(gdf[['Model', 'LL (ε=0)', 'Best LL ε', 'Best LL', 'LL Δ', 
               'Brier (ε=0)', 'Best Brier ε', 'Best Brier', 'Brier Δ',
               'Acc (ε=0)', 'Acc (best LL ε)']].to_string(index=False))

print("\n\nKey: ε=0 means no trimming is optimal (model is well-calibrated at extremes)")
print("Negative Δ = improvement from trimming, Positive Δ = trimming hurts")

PROBABILITY TRIMMING SUMMARY — Optimal ε per model (avg across spans 3/5/7)

  MEN'S BASKETBALL
  ------------------------------------------------------------------------------------------
              Model  LL (ε=0)  Best LL ε  Best LL    LL Δ  Brier (ε=0)  Best Brier ε  Best Brier  Brier Δ  Acc (ε=0)  Acc (best LL ε)
logistic_regression    0.5699        0.0   0.5699  0.0000       0.1946           0.0      0.1946   0.0000      69.71            69.71
                svm    0.5788        0.0   0.5788  0.0000       0.1981           0.0      0.1981   0.0000      69.26            69.26
                knn    0.6518        0.1   0.6239 -0.0279       0.2175           0.1      0.2174  -0.0001      64.69            64.69
      random_forest    0.5872        0.0   0.5872  0.0000       0.2014           0.0      0.2014   0.0000      68.54            68.54
  gradient_boosting    0.5755        0.0   0.5755  0.0000       0.1970           0.0      0.1970   0.0000      68.98            68.98
       

In [6]:
# Men's summary
m = summary_df[summary_df['Gender']=="Men's"][['Model','Best LL ε','LL Δ','Best Brier ε','Brier Δ']]
print("MEN'S:")
print(m.to_string(index=False))
print()
# Women's summary
w = summary_df[summary_df['Gender']=="Women's"][['Model','Best LL ε','LL Δ','Best Brier ε','Brier Δ']]
print("WOMEN'S:")
print(w.to_string(index=False))

MEN'S:
              Model  Best LL ε    LL Δ  Best Brier ε  Brier Δ
logistic_regression        0.0  0.0000           0.0   0.0000
                svm        0.0  0.0000           0.0   0.0000
                knn        0.1 -0.0279           0.1  -0.0001
      random_forest        0.0  0.0000           0.0   0.0000
  gradient_boosting        0.0  0.0000           0.0   0.0000
                mlp        0.0  0.0000           0.0   0.0000
            xgboost        0.0  0.0000           0.0   0.0000
           lightgbm        0.0  0.0000           0.0   0.0000
   ensemble_7_equal        0.0  0.0000           0.0   0.0000

WOMEN'S:
              Model  Best LL ε    LL Δ  Best Brier ε  Brier Δ
logistic_regression       0.01 -0.0001          0.01  -0.0000
                svm       0.01 -0.0001          0.01  -0.0000
                knn       0.05 -0.0510          0.05  -0.0001
      random_forest       0.00  0.0000          0.00   0.0000
  gradient_boosting       0.00  0.0000          0.00 

In [8]:
# === Neutral vs Home/Away Performance Breakdown ===
# Compare model accuracy, log loss, and brier score on neutral-site games vs home/away

model_files = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}

for gender_label, is_womens in [("MEN'S", False), ("WOMEN'S", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')
    
    print(f"\n{'='*90}")
    print(f"  NEUTRAL vs HOME/AWAY — {gender_label}")
    print(f"{'='*90}")
    
    # Collect results across all spans
    venue_rows = []
    
    for span in [3, 5, 7]:
        te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
        testing_df = pd.read_csv(te_path)
        test_true = testing_df.pop('Win')
        neutral_mask = testing_df['Neutral'] == 1
        
        models = {}
        for nm, fn in model_files.items():
            path = os.path.join(mdir, f'{span}span_{fn}')
            models[nm] = pickle.load(open(path, 'rb'))
        
        for name, model in models.items():
            y_proba = model.predict_proba(testing_df)[:, 1]
            y_pred = model.predict(testing_df)
            
            for venue, mask in [('Neutral', neutral_mask), ('Home/Away', ~neutral_mask)]:
                acc = accuracy_score(test_true[mask], y_pred[mask]) * 100
                ll = log_loss(test_true[mask], y_proba[mask])
                bs = brier_score_loss(test_true[mask], y_proba[mask])
                venue_rows.append({
                    'span': span, 'model': name, 'venue': venue,
                    'n': mask.sum(), 'accuracy': round(acc, 2),
                    'log_loss': round(ll, 4), 'brier_score': round(bs, 4)
                })
        
        # Also add ensemble (7 models, no KNN, equal weights)
        no_knn = {k: v for k, v in models.items() if k != 'knn'}
        ens_proba = np.mean([m.predict_proba(testing_df)[:, 1] for m in no_knn.values()], axis=0)
        ens_pred = (ens_proba >= 0.5).astype(int)
        
        for venue, mask in [('Neutral', neutral_mask), ('Home/Away', ~neutral_mask)]:
            acc = accuracy_score(test_true[mask], ens_pred[mask]) * 100
            ll = log_loss(test_true[mask], ens_proba[mask])
            bs = brier_score_loss(test_true[mask], ens_proba[mask])
            venue_rows.append({
                'span': span, 'model': 'ensemble_7_equal', 'venue': venue,
                'n': mask.sum(), 'accuracy': round(acc, 2),
                'log_loss': round(ll, 4), 'brier_score': round(bs, 4)
            })
    
    vdf = pd.DataFrame(venue_rows)
    
    # Average across spans, pivot by venue
    avg = vdf.groupby(['model', 'venue']).agg({
        'n': 'mean', 'accuracy': 'mean', 'log_loss': 'mean', 'brier_score': 'mean'
    }).round(4)
    
    neutral = avg.xs('Neutral', level='venue').rename(columns={
        'accuracy': 'Acc_N', 'log_loss': 'LL_N', 'brier_score': 'Brier_N', 'n': 'n_N'
    })
    home_away = avg.xs('Home/Away', level='venue').rename(columns={
        'accuracy': 'Acc_HA', 'log_loss': 'LL_HA', 'brier_score': 'Brier_HA', 'n': 'n_HA'
    })
    
    combined = neutral.join(home_away)
    combined['Acc_Δ'] = (combined['Acc_N'] - combined['Acc_HA']).round(2)
    combined['LL_Δ'] = (combined['LL_N'] - combined['LL_HA']).round(4)
    combined = combined.sort_values('LL_N')
    
    print(f"\n  Avg across spans (N=Neutral, HA=Home/Away, Δ=N minus HA)")
    print(f"  n_N ≈ {combined['n_N'].iloc[0]:.0f} games/span, n_HA ≈ {combined['n_HA'].iloc[0]:.0f} games/span")
    print()
    print(combined[['Acc_N', 'Acc_HA', 'Acc_Δ', 'LL_N', 'LL_HA', 'LL_Δ', 'Brier_N', 'Brier_HA']].to_string())
    
    # Per-span detail for the ensemble
    print(f"\n  Per-Span Detail — Ensemble (7, equal):")
    ens_detail = vdf[vdf['model'] == 'ensemble_7_equal'][['span', 'venue', 'n', 'accuracy', 'log_loss', 'brier_score']]
    print(ens_detail.to_string(index=False))


  NEUTRAL vs HOME/AWAY — MEN'S

  Avg across spans (N=Neutral, HA=Home/Away, Δ=N minus HA)
  n_N ≈ 1345 games/span, n_HA ≈ 5867 games/span

                       Acc_N   Acc_HA  Acc_Δ    LL_N   LL_HA    LL_Δ  Brier_N  Brier_HA
model                                                                                  
logistic_regression  66.9933  70.3733  -3.38  0.6108  0.5602  0.0506   0.2113    0.1906
ensemble_7_equal     66.6833  70.6300  -3.95  0.6142  0.5606  0.0536   0.2128    0.1906
xgboost              65.6833  70.4333  -4.75  0.6177  0.5625  0.0552   0.2143    0.1915
svm                  65.7767  70.0100  -4.23  0.6191  0.5692  0.0499   0.2151    0.1941
lightgbm             65.6033  70.4167  -4.81  0.6192  0.5631  0.0561   0.2150    0.1916
gradient_boosting    64.9767  69.9033  -4.93  0.6228  0.5646  0.0582   0.2164    0.1925
mlp                  64.9867  69.5333  -4.55  0.6295  0.5725  0.0570   0.2188    0.1956
random_forest        62.9867  69.8733  -6.89  0.6410  0.5748  0.066

In [9]:
# === Compact Neutral vs Home/Away Summary ===

model_files = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}

for gender_label, is_womens in [("MEN'S", False), ("WOMEN'S", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')
    rows = []
    
    for span in [3, 5, 7]:
        te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
        testing_df = pd.read_csv(te_path)
        test_true = testing_df.pop('Win')
        neutral_mask = testing_df['Neutral'] == 1
        
        models = {}
        for nm, fn in model_files.items():
            models[nm] = pickle.load(open(os.path.join(mdir, f'{span}span_{fn}'), 'rb'))
        
        all_models = list(models.items())
        no_knn = {k: v for k, v in models.items() if k != 'knn'}
        
        # Add ensemble
        ens_proba = np.mean([m.predict_proba(testing_df)[:, 1] for m in no_knn.values()], axis=0)
        
        for name, model in all_models:
            y_proba = model.predict_proba(testing_df)[:, 1]
            y_pred = model.predict(testing_df)
            for venue, mask in [('N', neutral_mask), ('HA', ~neutral_mask)]:
                rows.append({'span': span, 'model': name, 'venue': venue,
                    'acc': accuracy_score(test_true[mask], y_pred[mask]) * 100,
                    'll': log_loss(test_true[mask], y_proba[mask])})
        
        ens_pred = (ens_proba >= 0.5).astype(int)
        for venue, mask in [('N', neutral_mask), ('HA', ~neutral_mask)]:
            rows.append({'span': span, 'model': '>> ensemble_7', 'venue': venue,
                'acc': accuracy_score(test_true[mask], ens_pred[mask]) * 100,
                'll': log_loss(test_true[mask], ens_proba[mask])})
    
    df = pd.DataFrame(rows)
    avg = df.groupby(['model','venue']).agg({'acc':'mean','ll':'mean'}).round(4)
    n_df = avg.xs('N', level='venue').rename(columns={'acc':'Acc_Neutral','ll':'LL_Neutral'})
    ha_df = avg.xs('HA', level='venue').rename(columns={'acc':'Acc_HomeAway','ll':'LL_HomeAway'})
    c = n_df.join(ha_df)
    c['Acc_Δ'] = (c['Acc_Neutral'] - c['Acc_HomeAway']).round(2)
    c['LL_Δ'] = (c['LL_Neutral'] - c['LL_HomeAway']).round(4)
    c = c.sort_values('LL_Neutral')
    
    print(f"\n{gender_label} (avg across spans 3/5/7)")
    print(f"Positive Acc_Δ = better on neutral, Negative = better on home/away")
    print(c[['Acc_Neutral','Acc_HomeAway','Acc_Δ','LL_Neutral','LL_HomeAway','LL_Δ']].to_string())
    print()


MEN'S (avg across spans 3/5/7)
Positive Acc_Δ = better on neutral, Negative = better on home/away
                     Acc_Neutral  Acc_HomeAway  Acc_Δ  LL_Neutral  LL_HomeAway    LL_Δ
model                                                                                 
logistic_regression      66.9925       70.3723  -3.38      0.6108       0.5602  0.0506
>> ensemble_7            66.6823       70.6313  -3.95      0.6142       0.5606  0.0536
xgboost                  65.6826       70.4332  -4.75      0.6177       0.5625  0.0552
svm                      65.7756       70.0094  -4.23      0.6191       0.5692  0.0499
lightgbm                 65.6000       70.4148  -4.81      0.6192       0.5631  0.0561
gradient_boosting        64.9794       69.9024  -4.92      0.6229       0.5646  0.0583
mlp                      64.9859       69.5316  -4.55      0.6295       0.5724  0.0571
random_forest            62.9908       69.8771  -6.89      0.6410       0.5748  0.0662
knn                      57.565

In [10]:
# Re-print the compact table (variables already computed above)
# Men's
import warnings
warnings.filterwarnings('ignore')

model_files = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}

results = {}
for gender_label, is_womens in [("MEN", False), ("WOMEN", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')
    rows = []
    for span in [3, 5, 7]:
        te = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
        tdf = pd.read_csv(te); tt = tdf.pop('Win'); nm = tdf['Neutral']==1
        mdls = {n: pickle.load(open(os.path.join(mdir, f'{span}span_{f}'),'rb')) for n,f in model_files.items()}
        nk = {k:v for k,v in mdls.items() if k!='knn'}
        ep = np.mean([m.predict_proba(tdf)[:,1] for m in nk.values()], axis=0)
        for name, model in list(mdls.items()) + [('>> ensemble_7', None)]:
            yp = model.predict_proba(tdf)[:,1] if model else ep
            yd = model.predict(tdf) if model else (ep>=0.5).astype(int)
            for v, mk in [('N', nm), ('HA', ~nm)]:
                rows.append({'model':name,'venue':v,
                    'acc':accuracy_score(tt[mk],yd[mk])*100, 'll':log_loss(tt[mk],yp[mk])})
    df = pd.DataFrame(rows)
    a = df.groupby(['model','venue']).mean().round(4)
    n = a.xs('N',level='venue').rename(columns={'acc':'Acc_N','ll':'LL_N'})
    h = a.xs('HA',level='venue').rename(columns={'acc':'Acc_HA','ll':'LL_HA'})
    c = n.join(h); c['Acc_Δ']=(c.Acc_N-c.Acc_HA).round(2); c['LL_Δ']=(c.LL_N-c.LL_HA).round(4)
    results[gender_label] = c.sort_values('LL_N')

for g, c in results.items():
    print(f"{g} (avg spans 3/5/7) | Δ = Neutral minus Home/Away")
    print(c[['Acc_N','Acc_HA','Acc_Δ','LL_N','LL_HA','LL_Δ']].to_string())
    print()

MEN (avg spans 3/5/7) | Δ = Neutral minus Home/Away
                       Acc_N   Acc_HA  Acc_Δ    LL_N   LL_HA    LL_Δ
model                                                               
logistic_regression  66.9925  70.3723  -3.38  0.6108  0.5602  0.0506
>> ensemble_7        66.6823  70.6313  -3.95  0.6142  0.5606  0.0536
xgboost              65.6826  70.4332  -4.75  0.6177  0.5625  0.0552
svm                  65.7756  70.0094  -4.23  0.6191  0.5692  0.0499
lightgbm             65.6000  70.4148  -4.81  0.6192  0.5631  0.0561
gradient_boosting    64.9794  69.9024  -4.92  0.6229  0.5646  0.0583
mlp                  64.9859  69.5316  -4.55  0.6295  0.5724  0.0571
random_forest        62.9908  69.8771  -6.89  0.6410  0.5748  0.0662
knn                  57.5656  66.3039  -8.74  0.7214  0.6352  0.0862

WOMEN (avg spans 3/5/7) | Δ = Neutral minus Home/Away
                       Acc_N   Acc_HA  Acc_Δ    LL_N   LL_HA    LL_Δ
model                                                            

In [3]:
# === Export Ensemble Models ===
# Strategy: Voting (no KNN, equal weights) for both genders
# 7 models: LogReg, SVM, RF, GBM, MLP, XGBoost, LightGBM

import warnings
warnings.filterwarnings('ignore')

MODEL_FILES = {
    'logistic_regression': 'logistic_regression_model.pkl',
    'svm': 'support_vector_machine_model.pkl',
    'knn': 'knn_model.pkl',
    'random_forest': 'random_forest.pkl',
    'gradient_boosting': 'gradient_boosting.pkl',
    'mlp': 'multilayer_perceptron.pkl',
    'xgboost': 'xgboost.pkl',
    'lightgbm': 'lightgbm.pkl'
}

le = LabelEncoder()
le.fit([0, 1])

for gender_label, is_womens in [("Men's", False), ("Women's", True)]:
    gender = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')

    print(f"\n{'='*60}")
    print(f"  {gender_label} Basketball - Voting (no KNN, equal)")
    print(f"{'='*60}")

    for span in [3, 5, 7]:
        models = {}
        for name, fname in MODEL_FILES.items():
            path = os.path.join(mdir, f'{span}span_{fname}')
            models[name] = pickle.load(open(path, 'rb'))

        estimators = [(name, model) for name, model in models.items() if name != 'knn']

        vc = VotingClassifier(estimators=estimators, voting='soft')
        vc.estimators_ = [model for _, model in estimators]
        vc.le_ = le
        vc.classes_ = np.array([0, 1])

        te_path = os.path.abspath(f'../../data/dataset/{gender}/{span}span_testing_set.csv')
        testing_df = pd.read_csv(te_path)
        test_true = testing_df.pop('Win')
        y_pred = vc.predict(testing_df)
        y_proba = vc.predict_proba(testing_df)[:, 1]
        acc = accuracy_score(test_true, y_pred) * 100
        ll = log_loss(test_true, y_proba)
        bs = brier_score_loss(test_true, y_proba)

        out_path = os.path.join(mdir, f'{span}span_ensemble.pkl')
        pickle.dump(vc, open(out_path, 'wb'))

        vc2 = pickle.load(open(out_path, 'rb'))
        assert np.allclose(y_proba, vc2.predict_proba(testing_df)[:, 1])

        fsize = os.path.getsize(out_path) / (1024*1024)
        print(f"  Span {span}: Acc={acc:.2f}%  LL={ll:.4f}  Brier={bs:.4f}  |  {out_path} ({fsize:.1f} MB)")

print("\nDone! 6 ensemble models exported.")


  Men's Basketball - Voting (no KNN, equal)
  Span 3: Acc=70.10%  LL=0.5684  Brier=0.1939  |  /workspace/machine-learning/model/mens/3span_ensemble.pkl (179.9 MB)
  Span 5: Acc=69.94%  LL=0.5665  Brier=0.1932  |  /workspace/machine-learning/model/mens/5span_ensemble.pkl (162.9 MB)
  Span 7: Acc=69.61%  LL=0.5771  Brier=0.1972  |  /workspace/machine-learning/model/mens/7span_ensemble.pkl (151.9 MB)

  Women's Basketball - Voting (no KNN, equal)
  Span 3: Acc=72.93%  LL=0.5210  Brier=0.1748  |  /workspace/machine-learning/model/womens/3span_ensemble.pkl (148.6 MB)
  Span 5: Acc=73.73%  LL=0.5185  Brier=0.1739  |  /workspace/machine-learning/model/womens/5span_ensemble.pkl (135.9 MB)
  Span 7: Acc=74.23%  LL=0.5070  Brier=0.1698  |  /workspace/machine-learning/model/womens/7span_ensemble.pkl (82.2 MB)

Done! 6 ensemble models exported.


# Exported Ensemble Evaluation
Stats for the 6 exported VotingClassifier models, broken down by span, neutral vs home/away.

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')

rows = []

for gender_label, is_womens in [("Men", False), ("Women", True)]:
    g = 'women' if is_womens else 'men'
    mdir = os.path.abspath(f'../model/{"womens" if is_womens else "mens"}/')

    for span in [3, 5, 7]:
        te_path = os.path.abspath(f'../../data/dataset/{g}/{span}span_testing_set.csv')
        testing_df = pd.read_csv(te_path)
        test_true = testing_df.pop('Win')
        neutral_mask = testing_df['Neutral'] == 1

        path = os.path.join(mdir, f'{span}span_ensemble.pkl')
        model = pickle.load(open(path, 'rb'))
        y_proba = model.predict_proba(testing_df)[:, 1]
        y_pred = model.predict(testing_df)

        for venue, mask in [('All', slice(None)), ('Neutral', neutral_mask), ('Home/Away', ~neutral_mask)]:
            yt = test_true.values[mask] if isinstance(mask, slice) else test_true[mask].values
            yp = y_proba[mask]
            yd = y_pred[mask]

            rows.append({
                'Gender': gender_label, 'Span': span, 'Venue': venue, 'N': len(yt),
                'Accuracy': round(accuracy_score(yt, yd) * 100, 2),
                'Precision': round(precision_score(yt, yd, zero_division=0) * 100, 2),
                'Recall': round(recall_score(yt, yd, zero_division=0) * 100, 2),
                'F1': round(f1_score(yt, yd, zero_division=0) * 100, 2),
                'Log Loss': round(log_loss(yt, yp), 4),
                'Brier': round(brier_score_loss(yt, yp), 4),
            })

results = pd.DataFrame(rows)

for gender in ['Men', 'Women']:
    gdf = results[results['Gender'] == gender]
    print(f"\n{'='*95}")
    print(f"  {gender.upper()}'S BASKETBALL — Ensemble (VotingClassifier, 7 models, no KNN)")
    print(f"{'='*95}")
    print(gdf[['Span', 'Venue', 'N', 'Accuracy', 'Precision', 'Recall', 'F1', 'Log Loss', 'Brier']].to_string(index=False))

    # Averages across spans
    avg = gdf.groupby('Venue')[['Accuracy', 'Precision', 'Recall', 'F1', 'Log Loss', 'Brier']].mean().round(4)
    avg.loc[:, ['Accuracy', 'Precision', 'Recall', 'F1']] = avg[['Accuracy', 'Precision', 'Recall', 'F1']].round(2)
    print(f"\n  Average across spans:")
    print(f"  {avg.to_string()}")
    print()


  MEN'S BASKETBALL — Ensemble (VotingClassifier, 7 models, no KNN)
 Span     Venue    N  Accuracy  Precision  Recall    F1  Log Loss  Brier
    3       All 7966     70.10      71.73   82.27 76.64    0.5684 0.1939
    3   Neutral 1729     65.88      63.43   70.08 66.59    0.6207 0.2157
    3 Home/Away 6237     71.27      73.44   84.89 78.75    0.5539 0.1879
    5       All 7178     69.94      71.82   82.04 76.59    0.5665 0.1932
    5   Neutral 1274     67.50      66.62   72.57 69.47    0.6066 0.2096
    5 Home/Away 5904     70.46      72.69   83.72 77.82    0.5579 0.1897
    7       All 6492     69.61      70.79   82.66 76.27    0.5771 0.1972
    7   Neutral 1032     66.67      64.19   76.54 69.82    0.6152 0.2130
    7 Home/Away 5460     70.16      71.85   83.62 77.29    0.5699 0.1942

  Average across spans:
             Accuracy  Precision  Recall     F1  Log Loss   Brier
Venue                                                          
All           69.88      71.45   82.32  76.50  

In [2]:
# Men's neutral-site comparison across spans
mens_neutral = results[(results['Gender'] == 'Men') & (results['Venue'] == 'Neutral')]
print("MEN'S — Neutral-site only (March Madness scenario)")
print(mens_neutral[['Span', 'N', 'Accuracy', 'Log Loss', 'Brier']].to_string(index=False))

MEN'S — Neutral-site only (March Madness scenario)
 Span    N  Accuracy  Log Loss  Brier
    3 1729     65.88    0.6207 0.2157
    5 1274     67.50    0.6066 0.2096
    7 1032     66.67    0.6152 0.2130


In [3]:
# Women's — all venue breakdowns across spans
# First 2 rounds: top 16 seeds host (Home/Away matters)
# Sweet 16 onward: all neutral sites
womens = results[results['Gender'] == 'Women']

print("WOMEN'S — By Venue & Span")
print("=" * 75)
for venue in ['Neutral', 'Home/Away', 'All']:
    vdf = womens[womens['Venue'] == venue]
    print(f"\n  {venue}:")
    print(vdf[['Span', 'N', 'Accuracy', 'Log Loss', 'Brier']].to_string(index=False))

# Which span wins where?
print("\n" + "=" * 75)
print("BEST SPAN BY VENUE (lowest log loss):")
for venue in ['Neutral', 'Home/Away', 'All']:
    vdf = womens[womens['Venue'] == venue]
    best = vdf.loc[vdf['Log Loss'].idxmin()]
    print(f"  {venue:<12} → Span {int(best['Span'])}  (Acc={best['Accuracy']}%, LL={best['Log Loss']}, Brier={best['Brier']})")

WOMEN'S — By Venue & Span

  Neutral:
 Span    N  Accuracy  Log Loss  Brier
    3 1108     70.22    0.5521 0.1872
    5  909     67.99    0.5629 0.1931
    7  684     69.01    0.5626 0.1938

  Home/Away:
 Span    N  Accuracy  Log Loss  Brier
    3 6257     73.41    0.5155 0.1726
    5 5771     74.63    0.5115 0.1709
    7 5316     74.91    0.4999 0.1667

  All:
 Span    N  Accuracy  Log Loss  Brier
    3 7365     72.93    0.5210 0.1748
    5 6680     73.73    0.5185 0.1739
    7 6000     74.23    0.5070 0.1698

BEST SPAN BY VENUE (lowest log loss):
  Neutral      → Span 3  (Acc=70.22%, LL=0.5521, Brier=0.1872)
  Home/Away    → Span 7  (Acc=74.91%, LL=0.4999, Brier=0.1667)
  All          → Span 7  (Acc=74.23%, LL=0.507, Brier=0.1698)


In [4]:
# Full metrics for both genders — Neutral vs Home/Away
for gender in ['Men', 'Women']:
    gdf = results[(results['Gender'] == gender) & (results['Venue'] != 'All')]
    print(f"\n{gender.upper()}'S — Precision / Recall / F1 by Venue & Span")
    print("=" * 85)
    print(gdf[['Span', 'Venue', 'N', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
    
    avg = gdf.groupby('Venue')[['Accuracy', 'Precision', 'Recall', 'F1']].mean().round(2)
    print(f"\n  Avg across spans:")
    print(f"  {avg.to_string()}")
    print()


MEN'S — Precision / Recall / F1 by Venue & Span
 Span     Venue    N  Accuracy  Precision  Recall    F1
    3   Neutral 1729     65.88      63.43   70.08 66.59
    3 Home/Away 6237     71.27      73.44   84.89 78.75
    5   Neutral 1274     67.50      66.62   72.57 69.47
    5 Home/Away 5904     70.46      72.69   83.72 77.82
    7   Neutral 1032     66.67      64.19   76.54 69.82
    7 Home/Away 5460     70.16      71.85   83.62 77.29

  Avg across spans:
             Accuracy  Precision  Recall     F1
Venue                                        
Home/Away     70.63      72.66   84.08  77.95
Neutral       66.68      64.75   73.06  68.63


WOMEN'S — Precision / Recall / F1 by Venue & Span
 Span     Venue    N  Accuracy  Precision  Recall    F1
    3   Neutral 1108     70.22      68.53   73.77 71.05
    3 Home/Away 6257     73.41      76.28   80.41 78.29
    5   Neutral  909     67.99      65.43   74.61 69.72
    5 Home/Away 5771     74.63      76.11   82.19 79.03
    7   Neutral  684